# 7. Transformación Silver — Ventas (v2)

**Versión:** 2.0  

**Cambios respecto a v1:** se incorpora el rechazo de ventas con `fecha_venta` nula (`FECHA_VENTA_NULA`), detectadas durante el diagnóstico.  

**Objetivo:** depurar `bronze.ventas`, aplicar reglas de calidad e integridad referencial y generar las tablas Delta `silver.ventas` y `silver.ventas_rechazados`.

**Reglas acordadas:**
- Conservar una ocurrencia por `venta_id`; enviar los excedentes a rechazados.
- Rechazar ventas con `fecha_venta` nula.
- Validar `cliente_id`, `producto_id` y `almacen_id` contra las tablas maestras Bronze.
- Si una venta tiene varias referencias inválidas, rechazarla una sola vez y registrar todos los motivos.
- Reemplazar `descuento_pct` nulo por `0`, bajo el supuesto de negocio documentado en `revision/Reglas_Negocio_y_Calidad_Silver.md`.
- Conservar `_source_file` y `_ingestion_timestamp`.
- No modificar Bronze.

**Importante:** los duplicados detectados en el diagnóstico inicial eran pares idénticos. La selección de una ocurrencia se realiza con `row_number`; al ser filas duplicadas idénticas en los datos de negocio, la fila conservada es equivalente en contenido.

## 1. Importaciones

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *


## 2. Parámetros

In [ ]:
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")

containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

esquemaBronze = "bronze"
esquemaSilver = "silver"

ruta_silver = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/silver/"

tabla_ventas_bronze = f"{catalogo}.{esquemaBronze}.ventas"
tabla_clientes_bronze = f"{catalogo}.{esquemaBronze}.clientes"
tabla_productos_bronze = f"{catalogo}.{esquemaBronze}.productos"
tabla_almacenes_bronze = f"{catalogo}.{esquemaBronze}.almacenes"

tabla_ventas_silver = f"{catalogo}.{esquemaSilver}.ventas"
tabla_ventas_rechazados = f"{catalogo}.{esquemaSilver}.ventas_rechazados"

print("Catálogo:", catalogo)
print("Origen:", tabla_ventas_bronze)
print("Referencia clientes:", tabla_clientes_bronze)
print("Referencia productos:", tabla_productos_bronze)
print("Referencia almacenes:", tabla_almacenes_bronze)
print("Destino:", tabla_ventas_silver)
print("Rechazados:", tabla_ventas_rechazados)
print("Ruta Silver:", ruta_silver)


## 3. Lectura desde Bronze

In [ ]:
df_ventas_bronze = spark.table(tabla_ventas_bronze)
df_clientes_bronze = spark.table(tabla_clientes_bronze)
df_productos_bronze = spark.table(tabla_productos_bronze)
df_almacenes_bronze = spark.table(tabla_almacenes_bronze)

print("Ventas Bronze:", df_ventas_bronze.count())
print("Clientes Bronze:", df_clientes_bronze.count())
print("Productos Bronze:", df_productos_bronze.count())
print("Almacenes Bronze:", df_almacenes_bronze.count())

df_ventas_bronze.printSchema()


## 4. Diagnóstico de calidad en Ventas

In [ ]:
# Conteo de nulos por columna para inspeccionar la calidad recibida.
nulos_ventas = df_ventas_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_ventas_bronze.columns
])

display(nulos_ventas)

print("Grupos duplicados de venta_id:",
      df_ventas_bronze.groupBy("venta_id").count().filter(F.col("count") > 1).count())

print("Registros con descuento_pct nulo:",
      df_ventas_bronze.filter(F.col("descuento_pct").isNull()).count())


## 5. Identificación y tratamiento de duplicados

Se identifica una ocurrencia por `venta_id`. Las filas excedentes se envían a rechazados con el motivo `VENTA_ID_DUPLICADO`.

El diagnóstico inicial encontró 20 grupos duplicados y 20 registros excedentes. Como los pares revisados eran idénticos, conservar cualquiera de las ocurrencias equivalentes no cambia los datos de negocio.

In [ ]:
ventana_venta = Window.partitionBy("venta_id").orderBy(F.monotonically_increasing_id())

df_ventas_con_orden = df_ventas_bronze.withColumn(
    "_numero_ocurrencia",
    F.row_number().over(ventana_venta)
)

df_ventas_rechazados_duplicados = (
    df_ventas_con_orden
    .filter(F.col("_numero_ocurrencia") > 1)
    .drop("_numero_ocurrencia")
    .withColumn("motivo_rechazo", F.lit("VENTA_ID_DUPLICADO"))
    .withColumn("fecha_proceso", F.current_timestamp())
)

df_ventas_unicas = (
    df_ventas_con_orden
    .filter(F.col("_numero_ocurrencia") == 1)
    .drop("_numero_ocurrencia")
)

print("Ventas Bronze:", df_ventas_bronze.count())
print("Ventas duplicadas enviadas a rechazados:", df_ventas_rechazados_duplicados.count())
print("Ventas únicas para validar referencias:", df_ventas_unicas.count())


## 6. Validación de calidad e integridad referencial

Las claves se validan contra las tablas maestras **Bronze**. Esto distingue una clave realmente inexistente de un registro maestro que existe en Bronze, aunque haya sido rechazado en Silver por otra regla de calidad.

También se rechazan las ventas cuya `fecha_venta` sea nula, con el motivo `FECHA_VENTA_NULA`, porque la fecha es necesaria para el análisis temporal. El diagnóstico identificó 40 registros con esta condición.

Una venta que incumpla más de una regla se registra una sola vez en cuarentena y conserva todos los motivos en `motivo_rechazo`, separados por `|`.

In [ ]:
clientes_validos = df_clientes_bronze.select("cliente_id").distinct()
productos_validos = df_productos_bronze.select("producto_id").distinct()
almacenes_validos = df_almacenes_bronze.select("almacen_id").distinct()

df_ventas_con_referencias = (
    df_ventas_unicas
    .join(
        clientes_validos.withColumn("_cliente_existe", F.lit(True)),
        on="cliente_id",
        how="left"
    )
    .join(
        productos_validos.withColumn("_producto_existe", F.lit(True)),
        on="producto_id",
        how="left"
    )
    .join(
        almacenes_validos.withColumn("_almacen_existe", F.lit(True)),
        on="almacen_id",
        how="left"
    )
)

motivos_referenciales = F.concat_ws(
    "|",
    F.when(F.col("fecha_venta").isNull(), F.lit("FECHA_VENTA_NULA")),
    F.when(F.col("_cliente_existe").isNull(), F.lit("CLIENTE_NO_EXISTE")),
    F.when(F.col("_producto_existe").isNull(), F.lit("PRODUCTO_NO_EXISTE")),
    F.when(F.col("_almacen_existe").isNull(), F.lit("ALMACEN_NO_EXISTE"))
)

df_ventas_con_referencias = df_ventas_con_referencias.withColumn(
    "motivo_rechazo",
    motivos_referenciales
)

df_ventas_rechazados_referencias = (
    df_ventas_con_referencias
    .filter(F.col("motivo_rechazo") != "")
    .drop("_cliente_existe", "_producto_existe", "_almacen_existe")
    .withColumn("fecha_proceso", F.current_timestamp())
)

df_ventas_candidatas = (
    df_ventas_con_referencias
    .filter(F.col("motivo_rechazo") == "")
    .drop(
        "_cliente_existe",
        "_producto_existe",
        "_almacen_existe",
        "motivo_rechazo"
    )
)

print("Ventas únicas:", df_ventas_unicas.count())
print("Rechazadas por fecha nula o referencias inválidas:",
      df_ventas_rechazados_referencias.count())
print("Candidatas Silver:", df_ventas_candidatas.count())

display(
    df_ventas_rechazados_referencias
    .groupBy("motivo_rechazo")
    .count()
    .orderBy("motivo_rechazo")
)


## 7. Normalización de descuento y preparación Silver

Regla `VEN-005`: los valores nulos de `descuento_pct` se reemplazan por `0`. Es un supuesto de negocio documentado para este proyecto: NULL se interpreta como ausencia de descuento. Bronze conserva el valor original.

In [ ]:
conteo_descuentos_nulos_antes = df_ventas_candidatas.filter(
    F.col("descuento_pct").isNull()
).count()

df_ventas_silver = (
    df_ventas_candidatas
    .withColumn(
        "descuento_pct",
        F.coalesce(F.col("descuento_pct"), F.lit(0).cast("decimal(5,2)"))
    )
    .select(
        "venta_id",
        "fecha_venta",
        "cliente_id",
        "producto_id",
        "almacen_id",
        "cantidad",
        "precio_unitario",
        "descuento_pct",
        "metodo_pago",
        "_source_file",
        "_ingestion_timestamp"
    )
)

print("Candidatas con descuento nulo antes de normalizar:",
      conteo_descuentos_nulos_antes)
print("Registros preparados para Silver:", df_ventas_silver.count())
print("Nulos en descuento después de normalizar:",
      df_ventas_silver.filter(F.col("descuento_pct").isNull()).count())

df_ventas_silver.printSchema()


## 8. Consolidación de rechazados y conciliación previa

In [ ]:
# Ambas fuentes de rechazo comparten los campos de origen.
# unionByName permite alinear columnas y completar con NULL cuando una regla no aplica.
df_ventas_rechazados = (
    df_ventas_rechazados_duplicados
    .unionByName(df_ventas_rechazados_referencias, allowMissingColumns=True)
)

conteo_bronze = df_ventas_bronze.count()
conteo_silver = df_ventas_silver.count()
conteo_rechazados = df_ventas_rechazados.count()

print("Ventas Bronze:", conteo_bronze)
print("Ventas Silver candidatas:", conteo_silver)
print("Ventas rechazadas:", conteo_rechazados)
print("Total conciliado:", conteo_silver + conteo_rechazados)

assert conteo_bronze == conteo_silver + conteo_rechazados, \
    "La conciliación previa no coincide con Bronze."

print("Conciliación previa correcta.")


## 9. Inspección antes de escribir

In [ ]:
display(df_ventas_silver.orderBy("venta_id").limit(20))


In [ ]:
display(df_ventas_rechazados.orderBy("venta_id").limit(20))


## 10. Preparar esquema y escribir tablas Delta

Ejecutar estas celdas después de revisar las validaciones y las muestras. `overwrite` reconstruye las tablas Silver si se vuelve a ejecutar el proceso; no modifica Bronze.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquemaSilver}")
print(f"Esquema disponible: {catalogo}.{esquemaSilver}")


In [ ]:
(
    df_ventas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_silver}ventas/")
    .saveAsTable(tabla_ventas_silver)
)

print("Tabla creada:", tabla_ventas_silver)


In [ ]:
(
    df_ventas_rechazados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_silver}ventas_rechazados/")
    .saveAsTable(tabla_ventas_rechazados)
)

print("Tabla creada:", tabla_ventas_rechazados)


## 11. Validación final de tablas persistidas

In [ ]:
df_ventas_silver_tabla = spark.table(tabla_ventas_silver)
df_ventas_rechazados_tabla = spark.table(tabla_ventas_rechazados)

conteo_bronze_final = df_ventas_bronze.count()
conteo_silver_final = df_ventas_silver_tabla.count()
conteo_rechazados_final = df_ventas_rechazados_tabla.count()

print("Bronze:", conteo_bronze_final)
print("Silver:", conteo_silver_final)
print("Rechazados:", conteo_rechazados_final)
print("Total conciliado:", conteo_silver_final + conteo_rechazados_final)
print("Descuentos NULL en Silver:",
      df_ventas_silver_tabla.filter(F.col("descuento_pct").isNull()).count())

assert conteo_bronze_final == conteo_silver_final + conteo_rechazados_final, \
    "La conciliación final no coincide con Bronze."

assert df_ventas_silver_tabla.groupBy("venta_id").count().filter(
    F.col("count") > 1
).count() == 0, "Se encontraron venta_id duplicados en Silver."

assert df_ventas_silver_tabla.filter(
    F.col("descuento_pct").isNull()
).count() == 0, "Quedaron descuentos NULL en Silver."

assert df_ventas_silver_tabla.filter(
    F.col("fecha_venta").isNull()
).count() == 0, "Quedaron fechas NULL en Silver."

print("Validación final correcta.")


## Resultado esperado según el diagnóstico inicial

- Bronze: 100,020 registros.
- Duplicados excedentes por `venta_id`: 20.
- `fecha_venta` nula: 40 ventas; se rechazan con `FECHA_VENTA_NULA` (regla incorporada en v2).
- Referencias inválidas detectadas inicialmente: 15 clientes y 25 productos; pueden coincidir en una misma venta.
- Almacenes inexistentes: 0.
- `descuento_pct` nulos: 100; se normalizan a 0 en los registros que llegan a Silver.
- Los conteos definitivos de Silver y rechazados se confirmarán durante la ejecución.

**Nota:** la conciliación debe contar cada fila de origen una sola vez. Las reglas de duplicidad se aplican primero; luego se validan las referencias sobre las ventas únicas.